# Day03：淘宝商品数据Pandas探索

**姓名：** 胡斌衔

本Notebook由每名学生独立完成，并提交到个人GitHub仓库。

> 请完成所有TODO和检查点。不要覆盖原始数据文件。

## 实验目标

你需要完成：

1. 读取25,000条淘宝商品记录；
2. 检查字段类型和缺失值；
3. 完成列选择、行选择、条件筛选和排序；
4. 完成商品价格及一级品类统计；
5. 完成“省份—类别”挑战分析；
6. 输出两张统计表并撰写有边界的结论。

### 数据边界

- 一行代表一条商品记录；
- `商品id`是标识符，不适合求平均值；
- `商品销量`包含“100+人付款”“1万+人付款”等文本，本阶段不直接求平均；
- `商品价格`是商品标价，不代表实际成交金额。

## 任务0：环境与个人配置

In [6]:
# ==================== 任务1：读取数据并完成初步观察 ====================
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置显示选项
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 1. 配置个人信息
STUDENT_NAME = "胡斌衔"  # 请替换为您的真实姓名

# 2. 查找项目根目录
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "data" / "淘宝全品类全国数据.csv").exists():
            return candidate
    raise FileNotFoundError("未找到 data/淘宝全品类全国数据.csv")

ROOT = find_project_root()
DATA_PATH = ROOT / "data" / "淘宝全品类全国数据.csv"

print("="*60)
print("任务1：读取数据并完成初步观察")
print("="*60)
print(f"学生姓名：{STUDENT_NAME}")
print(f"数据路径：{DATA_PATH}")
print(f"数据存在：{DATA_PATH.exists()}")
print("="*60)

# ==================== 1.1 读取数据 ====================
print("\\n【1.1 读取数据】")
print("-"*40)

def load_data_with_encoding():
    """尝试不同编码读取数据"""
    encodings = ['utf-8', 'gbk', 'gb2312', 'gb18030', 'latin1']

    for encoding in encodings:
        try:
            df = pd.read_csv(DATA_PATH, encoding=encoding)
            print(f"✅ 成功使用 {encoding} 编码读取数据")
            return df, encoding
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"❌ 使用 {encoding} 编码读取失败：{e}")
            continue

    raise Exception("所有编码尝试均失败，请检查文件")

# 读取数据
df, used_encoding = load_data_with_encoding()

# ==================== 1.2 初步观察 ====================
print("\\n【1.2 初步观察】")
print("-"*40)

# 1.2.1 查看数据形状
print(f"\\n1.2.1 数据形状：")
print(f"  行数（样本数）：{df.shape[0]:,}")
print(f"  列数（特征数）：{df.shape[1]}")

# 1.2.2 查看列名
print(f"\\n1.2.2 所有列名（共{len(df.columns)}列）：")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

# 1.2.3 查看数据类型
print(f"\\n1.2.3 数据类型信息：")
dtype_summary = df.dtypes.value_counts()
for dtype, count in dtype_summary.items():
    print(f"  {dtype}: {count}列")

# 详细数据类型
print(f"\\n详细数据类型：")
for col in df.columns:
    print(f"  {col:20s} : {df[col].dtype}")

# 1.2.4 查看前5行
print(f"\\n1.2.4 前5行数据：")
print(df.head())

# 1.2.5 查看后5行
print(f"\\n1.2.5 后5行数据：")
print(df.tail())

# 1.2.6 随机抽样5行
print(f"\\n1.2.6 随机抽样5行数据：")
print(df.sample(5))

# ==================== 1.3 数据基本信息统计 ====================
print("\\n【1.3 数据基本信息统计】")
print("-"*40)

# 1.3.1 内存使用
memory_usage = df.memory_usage(deep=True)
print(f"\\n1.3.1 内存使用：")
print(f"  总内存使用：{memory_usage.sum() / 1024 / 1024:.2f} MB")
print(f"  平均每行：{memory_usage.sum() / len(df) / 1024:.2f} KB")

# 1.3.2 数据完整性
print(f"\\n1.3.2 数据完整性：")
non_null_counts = df.count()
null_counts = df.isnull().sum()
null_percent = (null_counts / len(df)) * 100

completeness_df = pd.DataFrame({
    '非空数量': non_null_counts,
    '缺失数量': null_counts,
    '缺失比例(%)': null_percent
})
print(completeness_df)

# 找出缺失的列
missing_cols = null_counts[null_counts > 0]
if len(missing_cols) > 0:
    print(f"\\n有缺失值的列（共{len(missing_cols)}列）：")
    for col in missing_cols.index:
        print(f"  {col}: 缺失{missing_cols[col]:,}条 ({null_percent[col]:.2f}%)")
else:
    print("\\n✅ 所有列均无缺失值")

# 1.3.3 重复值检查
duplicates = df.duplicated()
print(f"\\n1.3.3 重复值检查：")
print(f"  重复行数：{duplicates.sum():,}")
print(f"  重复比例：{(duplicates.sum() / len(df)) * 100:.2f}%")

# 1.3.4 唯一值统计
print(f"\\n1.3.4 各列唯一值数量：")
unique_counts = df.nunique()
for col in df.columns:
    print(f"  {col:20s} : {unique_counts[col]:,} 个唯一值")

# ==================== 1.4 数值列统计分析 ====================
print("\\n【1.4 数值列统计分析】")
print("-"*40)

# 识别数值列
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(f"\\n数值列（共{len(numeric_cols)}列）：")
for col in numeric_cols:
    print(f"  {col}")

if len(numeric_cols) > 0:
    # 统计描述
    print(f"\\n数值列统计描述：")
    print(df[numeric_cols].describe())

    # 更详细的统计信息
    print(f"\\n数值列详细统计：")
    stats_df = pd.DataFrame({
        '均值': df[numeric_cols].mean(),
        '中位数': df[numeric_cols].median(),
        '标准差': df[numeric_cols].std(),
        '最小值': df[numeric_cols].min(),
        '最大值': df[numeric_cols].max(),
        '峰度': df[numeric_cols].kurtosis(),
        '偏度': df[numeric_cols].skew()
    })
    print(stats_df)

    # 识别异常值（使用IQR方法）
    print(f"\\n异常值检测（IQR方法）：")
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        outlier_count = len(outliers)
        outlier_percent = (outlier_count / len(df)) * 100

        if outlier_count > 0:
            print(f"  {col}: {outlier_count:,} 个异常值 ({outlier_percent:.2f}%)")
            print(f"    正常范围: [{lower_bound:.2f}, {upper_bound:.2f}]")
else:
    print("⚠️ 未发现数值列")

# ==================== 1.5 分类列统计分析 ====================
print("\\n【1.5 分类列统计分析】")
print("-"*40)

# 识别分类列
categorical_cols = df.select_dtypes(include=['object']).columns
print(f"\\n分类列（共{len(categorical_cols)}列）：")
for col in categorical_cols:
    print(f"  {col}")

if len(categorical_cols) > 0:
    for col in categorical_cols:
        print(f"\\n列 '{col}' 的分布：")
        value_counts = df[col].value_counts()
        top_n = min(10, len(value_counts))

        print(f"  总类别数：{len(value_counts)}")
        print(f"  前{top_n}个类别：")

        # 计算百分比
        total = len(df)
        for value, count in value_counts.head(top_n).items():
            percent = (count / total) * 100
            print(f"    {value}: {count:,} ({percent:.2f}%)")

        if len(value_counts) > top_n:
            other_count = value_counts[top_n:].sum()
            other_percent = (other_count / total) * 100
            print(f"    其他: {other_count:,} ({other_percent:.2f}%)")

        # 缺失值统计
        null_count = df[col].isnull().sum()
        if null_count > 0:
            print(f"  缺失值：{null_count:,} ({null_count/total*100:.2f}%)")
else:
    print("⚠️ 未发现分类列")

# ==================== 1.6 数据预览总结 ====================
print("\\n【1.6 数据预览总结】")
print("-"*40)

summary = {
    '数据集名称': '淘宝全品类全国数据',
    '总行数': df.shape[0],
    '总列数': df.shape[1],
    '数值列数': len(numeric_cols),
    '分类列数': len(categorical_cols),
    '缺失值总数': df.isnull().sum().sum(),
    '缺失值比例': (df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100,
    '重复行数': duplicates.sum(),
    '重复比例': (duplicates.sum() / len(df)) * 100,
    '内存使用(MB)': memory_usage.sum() / 1024 / 1024,
    '使用编码': used_encoding
}

print("数据概览：")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"  {key:12s}: {value:.2f}")
    else:
        print(f"  {key:12s}: {value}")

# ==================== 1.7 保存初步观察结果 ====================
print("\\n【1.7 保存初步观察结果】")
print("-"*40)

# 创建输出目录
output_dir = ROOT / "output" / "day03_analysis"
output_dir.mkdir(parents=True, exist_ok=True)

# 保存统计信息到CSV
# 1. 数值列统计
if len(numeric_cols) > 0:
    stats_df = df[numeric_cols].describe()
    stats_df.to_csv(output_dir / "numeric_stats.csv")
    print("✅ 数值列统计已保存：numeric_stats.csv")

# 2. 缺失值统计
null_df = pd.DataFrame({
    '列名': null_counts.index,
    '缺失数量': null_counts.values,
    '缺失比例(%)': null_percent.values
})
null_df.to_csv(output_dir / "missing_values.csv", index=False)
print("✅ 缺失值统计已保存：missing_values.csv")

# 3. 数据类型信息
dtype_df = pd.DataFrame({
    '列名': df.columns,
    '数据类型': df.dtypes.values,
    '非空数量': non_null_counts.values,
    '唯一值数量': unique_counts.values
})
dtype_df.to_csv(output_dir / "data_types.csv", index=False)
print("✅ 数据类型信息已保存：data_types.csv")

# 4. 保存数据概览报告
with open(output_dir / "数据初步观察报告.txt", 'w', encoding='utf-8') as f:
    f.write("="*60 + "\\n")
    f.write("淘宝全品类全国数据 - 初步观察报告\\n")
    f.write("="*60 + "\\n\\n")
    f.write(f"分析人：{STUDENT_NAME}\\n")
    f.write(f"数据路径：{DATA_PATH}\\n")
    f.write(f"报告生成时间：{pd.Timestamp.now()}\\n\\n")

    f.write("【数据概览】\\n")
    f.write("-"*40 + "\\n")
    for key, value in summary.items():
        if isinstance(value, float):
            f.write(f"{key:15s}: {value:.2f}\\n")
        else:
            f.write(f"{key:15s}: {value}\\n")

    f.write("\\n【列信息】\\n")
    f.write("-"*40 + "\\n")
    for i, col in enumerate(df.columns, 1):
        f.write(f"{i:3d}. {col:25s} {df[col].dtype}\\n")

    if len(missing_cols) > 0:
        f.write("\\n【缺失值列】\\n")
        f.write("-"*40 + "\\n")
        for col in missing_cols.index:
            f.write(f"{col:25s}: {missing_cols[col]:,} ({null_percent[col]:.2f}%)\\n")

    if duplicates.sum() > 0:
        f.write(f"\\n【重复数据】\\n")
        f.write("-"*40 + "\\n")
        f.write(f"重复行数：{duplicates.sum():,}\\n")
        f.write(f"重复比例：{(duplicates.sum() / len(df)) * 100:.2f}%\\n")

print("✅ 数据初步观察报告已保存：数据初步观察报告.txt")

print("\\n" + "="*60)
print("任务1完成！")
print("="*60)
print(f"所有输出文件已保存至：{output_dir}")
print(f"生成文件列表：")
for file in output_dir.iterdir():
    print(f"  - {file.name}")
print("="*60)

任务1：读取数据并完成初步观察
学生姓名：胡斌衔
数据路径：D:\training2\ecommerce-user-analysis-seed (1）\muc-commerce-3-24040087\data\淘宝全品类全国数据.csv
数据存在：True
\n【1.1 读取数据】
----------------------------------------
✅ 成功使用 utf-8 编码读取数据
\n【1.2 初步观察】
----------------------------------------
\n1.2.1 数据形状：
  行数（样本数）：25,000
  列数（特征数）：15
\n1.2.2 所有列名（共15列）：
   1. 商品id
   2. 一级品类
   3. 二级品类
   4. 商品标题
   5. 商品价格
   6. 省份
   7. 商品销量
   8. 店铺名称
   9. 店铺标签
  10. 先用后付
  11. 退货宝
  12. 风格
  13. 面料
  14. 版型
  15. 适用季节
\n1.2.3 数据类型信息：
  str: 14列
  float64: 1列
\n详细数据类型：
  商品id                 : str
  一级品类                 : str
  二级品类                 : str
  商品标题                 : str
  商品价格                 : float64
  省份                   : str
  商品销量                 : str
  店铺名称                 : str
  店铺标签                 : str
  先用后付                 : str
  退货宝                  : str
  风格                   : str
  面料                   : str
  版型                   : str
  适用季节                 : str
\n1.2.4 前5行数据：
             商品id 

## 任务1：读取数据并完成初步观察

In [4]:
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
print("字段名：", df.columns.tolist())
display(df.head())

数据形状： (25000, 15)
字段名： ['商品id', '一级品类', '二级品类', '商品标题', '商品价格', '省份', '商品销量', '店铺名称', '店铺标签', '先用后付', '退货宝', '风格', '面料', '版型', '适用季节']


,商品id,一级品类,二级品类,商品标题,商品价格,省份,商品销量,店铺名称,店铺标签,先用后付,退货宝,风格,面料,版型,适用季节
0,\t446974700314,汽车用品,保养,保养2025新款,980.47,广东,500+人付款,粤优品汽车店,5年老店,NaN,NaN,NaN,NaN,NaN,NaN
1,\t960353038337,食品生鲜,粮油,粮油官方正品,344.47,北京,100+人付款,诚信食品店,皇冠店铺,NaN,NaN,NaN,NaN,NaN,NaN
2,\t765651339105,图书音像,教材,教材2025新款,261.81,香港,1000+人付款,港优品图书店,8年老店,先用后付,NaN,NaN,NaN,NaN,NaN
3,\t614914975025,服饰鞋包,童装,童装修身2025新款,503.53,天津,2000+人付款,津优品服饰店专卖店,NaN,NaN,NaN,休闲风,羊毛,标准型,春秋季
4,\t757714643103,家居生活,装饰,装饰官方正品户外,"1,282.75",北京,500+人付款,时尚家居店旗舰店,回头客3千,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# 检查点1
assert df.shape == (25000, 15), "数据形状应为(25000, 15)"
assert {"商品id", "一级品类", "商品价格", "省份", "商品销量"}.issubset(df.columns)
print("检查点1通过")

检查点1通过


**数据粒度：** 请用一句话说明一行代表什么。

> TODO：一行代表每一个独立用户的购买记录。

## 任务2：字段类型与缺失值

In [7]:
# ==================== 任务2：字段类型与缺失值（修复版） ====================
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置显示选项
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 1. 配置个人信息
STUDENT_NAME = "胡斌衔"  # 请替换为您的真实姓名

# 2. 查找项目根目录并加载数据
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "data" / "淘宝全品类全国数据.csv").exists():
            return candidate
    raise FileNotFoundError("未找到 data/淘宝全品类全国数据.csv")

ROOT = find_project_root()
DATA_PATH = ROOT / "data" / "淘宝全品类全国数据.csv"

def load_data_with_encoding():
    encodings = ['utf-8', 'gbk', 'gb2312', 'gb18030', 'latin1']
    for encoding in encodings:
        try:
            df = pd.read_csv(DATA_PATH, encoding=encoding)
            print(f"✅ 成功使用 {encoding} 编码读取数据")
            return df
        except:
            continue
    raise Exception("所有编码尝试均失败")

df = load_data_with_encoding()

print("="*60)
print("任务2：字段类型与缺失值")
print("="*60)
print(f"学生姓名：{STUDENT_NAME}")
print(f"数据形状：{df.shape}")
print("="*60)

# ==================== TODO 1：输出字段类型 ====================
print("\\n【TODO 1：输出字段类型】")
print("-"*40)

# 方式1：使用dtypes属性
print("方式1 - 所有字段类型：")
print(df.dtypes)

# 方式2：更详细的信息（包括非空数量等）
print("\\n方式2 - 详细信息：")
print(df.info())

# 方式3：分类统计
print("\\n方式3 - 类型分类统计：")
dtype_counts = df.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f"  {dtype}: {count}列")

# 方式4：按类型分组显示列名
print("\\n方式4 - 按类型分组：")
for dtype in df.dtypes.unique():
    cols = df.dtypes[df.dtypes == dtype].index.tolist()
    print(f"\\n{dtype}类型列（共{len(cols)}列）：")
    for i, col in enumerate(cols, 1):
        print(f"  {i:2d}. {col}")

# 方式5：字段类型汇总表（修复版本）
print("\\n方式5 - 字段类型汇总表：")
# 修复：确保所有列表长度一致
dtype_summary = pd.DataFrame({
    '列名': df.columns.tolist(),
    '数据类型': df.dtypes.values.tolist(),
    '非空数量': df.count().values.tolist(),
    '缺失数量': df.isnull().sum().values.tolist(),
    '缺失率(%)': (df.isnull().sum() / len(df) * 100).values.tolist()
})
print(dtype_summary)

# ==================== TODO 2：计算缺失数量并从高到低排序 ====================
print("\\n【TODO 2：计算缺失数量并从高到低排序】")
print("-"*40)

# 计算每列的缺失数量，并从高到低排序
missing_count = df.isnull().sum().sort_values(ascending=False)

# 输出结果
print("缺失数量（从高到低排序）：")
for col, count in missing_count.items():
    if count > 0:
        print(f"  {col:25s}: {count:6,} 条缺失")
    else:
        print(f"  {col:25s}: {count:6,} 条缺失 ✅")

# 只显示有缺失值的列
if missing_count.sum() > 0:
    print(f"\\n有缺失值的列（共{len(missing_count[missing_count > 0])}列）：")
    for col, count in missing_count[missing_count > 0].items():
        print(f"  {col:25s}: {count:6,} 条缺失")
else:
    print("\\n✅ 所有列均无缺失值！")

# ==================== TODO 3：计算缺失率（百分比） ====================
print("\\n【TODO 3：计算缺失率（百分比）】")
print("-"*40)

# 计算每列的缺失率（百分比），从高到低排序
missing_rate = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

# 输出结果
print("缺失率（从高到低排序）：")
for col, rate in missing_rate.items():
    if rate > 0:
        print(f"  {col:25s}: {rate:6.2f}%")
    else:
        print(f"  {col:25s}: {rate:6.2f}% ✅")

# 只显示有缺失值的列
if missing_rate.sum() > 0:
    print(f"\\n有缺失值的列（共{len(missing_rate[missing_rate > 0])}列）：")
    for col, rate in missing_rate[missing_rate > 0].items():
        count = df[col].isnull().sum()
        print(f"  {col:25s}: {rate:6.2f}% ({count:,}条)")
else:
    print("\\n✅ 所有列均无缺失值！")

# ==================== TODO 4：展示结果 ====================
print("\\n【TODO 4：展示结果】")
print("-"*40)

# 方式1：合并展示（推荐）
print("方式1 - 合并展示（缺失数量 + 缺失率）：")
# 修复：确保索引一致
missing_summary = pd.DataFrame({
    '缺失数量': missing_count.values,
    '缺失率(%)': missing_rate.values
}, index=missing_count.index)
print(missing_summary)

# 方式2：只显示有缺失值的列
if missing_count.sum() > 0:
    print("\\n方式2 - 仅显示有缺失值的列：")
    missing_cols = missing_count[missing_count > 0]
    missing_summary_filtered = pd.DataFrame({
        '缺失数量': missing_cols.values,
        '缺失率(%)': missing_rate[missing_cols.index].values
    }, index=missing_cols.index)
    print(missing_summary_filtered)
else:
    print("\\n方式2 - 所有列均无缺失值")

# 方式3：按缺失率分组展示
print("\\n方式3 - 按缺失率分组：")
rate_groups = {
    '高缺失率(>50%)': missing_rate[missing_rate > 50],
    '中缺失率(10%-50%)': missing_rate[(missing_rate >= 10) & (missing_rate <= 50)],
    '低缺失率(<10%)': missing_rate[(missing_rate > 0) & (missing_rate < 10)],
    '无缺失(0%)': missing_rate[missing_rate == 0]
}

for group_name, group_data in rate_groups.items():
    if len(group_data) > 0:
        print(f"\\n{group_name}（共{len(group_data)}列）：")
        for col, rate in group_data.items():
            if rate > 0:
                count = df[col].isnull().sum()
                print(f"    {col:25s}: {rate:.2f}% ({count:,}条)")
            else:
                print(f"    {col:25s}: {rate:.2f}%")

# 方式4：可视化展示（文本版）
print("\\n方式4 - 缺失值条形图（文本版）：")
max_len = 50  # 最大条长度
max_count = missing_count.max() if missing_count.max() > 0 else 1

for col, count in missing_count.items():
    if count > 0:
        # 计算条长度
        bar_len = int((count / max_count) * max_len)
        bar = '█' * bar_len
        percentage = (count / len(df)) * 100
        print(f"{col:20s} |{bar:<{max_len}} {count:6,} ({percentage:5.2f}%)")
    else:
        print(f"{col:20s} |{' ' * max_len} {count:6,} (0.00%)")

# ==================== 额外：数据完整性评估 ====================
print("\\n【数据完整性评估】")
print("-"*40)

total_cells = df.shape[0] * df.shape[1]
missing_cells = df.isnull().sum().sum()
missing_percent = (missing_cells / total_cells) * 100

print(f"总单元格数：{total_cells:,}")
print(f"缺失单元格数：{missing_cells:,}")
print(f"缺失率：{missing_percent:.2f}%")
print(f"完整率：{(100 - missing_percent):.2f}%")

# 完整性评分
if missing_percent == 0:
    print("✅ 数据完整性评级：A+ (完美)")
elif missing_percent < 5:
    print("✅ 数据完整性评级：A (优秀)")
elif missing_percent < 10:
    print("✅ 数据完整性评级：B (良好)")
elif missing_percent < 20:
    print("⚠️ 数据完整性评级：C (一般，需要处理缺失值)")
else:
    print("❌ 数据完整性评级：D (较差，建议重新收集或大量填充)")

# ==================== 保存结果 ====================
print("\\n【保存结果】")
print("-"*40)

# 创建输出目录
output_dir = ROOT / "output" / "day03_analysis"
output_dir.mkdir(parents=True, exist_ok=True)

# 保存缺失值分析结果（修复版本）
missing_summary_save = pd.DataFrame({
    '缺失数量': missing_count.values,
    '缺失率(%)': missing_rate.values
}, index=missing_count.index)
missing_summary_save.to_csv(output_dir / "missing_analysis.csv", encoding='utf-8-sig')
print(f"✅ 缺失值分析已保存：{output_dir}/missing_analysis.csv")

# 保存字段类型信息（修复版本）
dtype_summary_save = pd.DataFrame({
    '列名': df.columns.tolist(),
    '数据类型': df.dtypes.values.tolist(),
    '非空数量': df.count().values.tolist(),
    '缺失数量': df.isnull().sum().values.tolist(),
    '缺失率(%)': (df.isnull().sum() / len(df) * 100).values.tolist()
})
dtype_summary_save.to_csv(output_dir / "field_types.csv", index=False, encoding='utf-8-sig')
print(f"✅ 字段类型信息已保存：{output_dir}/field_types.csv")

# 生成文本报告
with open(output_dir / "字段类型与缺失值报告.txt", 'w', encoding='utf-8') as f:
    f.write("="*60 + "\\n")
    f.write("任务2：字段类型与缺失值分析报告\\n")
    f.write("="*60 + "\\n\\n")
    f.write(f"分析人：{STUDENT_NAME}\\n")
    f.write(f"分析时间：{pd.Timestamp.now()}\\n")
    f.write(f"数据文件：{DATA_PATH.name}\\n")
    f.write(f"数据形状：{df.shape}\\n\\n")

    f.write("【一、字段类型】\\n")
    f.write("-"*40 + "\\n")
    for col in df.columns:
        f.write(f"{col:25s} : {df[col].dtype}\\n")

    f.write("\\n【二、缺失值统计】\\n")
    f.write("-"*40 + "\\n")
    f.write(f"总缺失值：{missing_cells:,} ({missing_percent:.2f}%)\\n\\n")

    if missing_count.sum() > 0:
        f.write("各列缺失情况（从高到低）：\\n")
        for col, count in missing_count[missing_count > 0].items():
            rate = (count / len(df)) * 100
            f.write(f"  {col:25s} : {count:6,} ({rate:5.2f}%)\\n")
    else:
        f.write("✅ 所有列均无缺失值！\\n")

    f.write("\\n【三、结论】\\n")
    f.write("-"*40 + "\\n")
    if missing_percent == 0:
        level = "A+ (完美)"
    elif missing_percent < 5:
        level = "A (优秀)"
    elif missing_percent < 10:
        level = "B (良好)"
    elif missing_percent < 20:
        level = "C (一般)"
    else:
        level = "D (较差)"
    f.write(f"数据完整性等级：{level}\\n")

    if missing_percent > 0:
        f.write("\\n建议：\\n")
        if missing_percent < 5:
            f.write("  缺失值较少，可以选择直接删除缺失行或简单填充\\n")
        elif missing_percent < 10:
            f.write("  缺失值适中，建议使用中位数/众数/均值填充\\n")
        elif missing_percent < 20:
            f.write("  缺失值较多，建议分析缺失模式，使用合适的填充策略\\n")
        else:
            f.write("  缺失值严重，建议重新评估数据质量或使用高级填充方法\\n")

print(f"✅ 文本报告已保存：{output_dir}/字段类型与缺失值报告.txt")

print("\\n" + "="*60)
print("任务2完成！")
print("="*60)
print(f"输出文件：")
for file in output_dir.iterdir():
    print(f"  - {file.name}")
print("="*60)

✅ 成功使用 utf-8 编码读取数据
任务2：字段类型与缺失值
学生姓名：胡斌衔
数据形状：(25000, 15)
\n【TODO 1：输出字段类型】
----------------------------------------
方式1 - 所有字段类型：
商品id        str
一级品类        str
二级品类        str
商品标题        str
商品价格    float64
省份          str
商品销量        str
店铺名称        str
店铺标签        str
先用后付        str
退货宝         str
风格          str
面料          str
版型          str
适用季节        str
dtype: object
\n方式2 - 详细信息：
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 15 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   商品id    25000 non-null  str    
 1   一级品类    25000 non-null  str    
 2   二级品类    25000 non-null  str    
 3   商品标题    25000 non-null  str    
 4   商品价格    25000 non-null  float64
 5   省份      25000 non-null  str    
 6   商品销量    25000 non-null  str    
 7   店铺名称    25000 non-null  str    
 8   店铺标签    21259 non-null  str    
 9   先用后付    3158 non-null   str    
 10  退货宝     2724 non-null   str    
 11  风格      3988 non

In [8]:
# 检查点2
assert isinstance(missing_count, pd.Series), "missing_count应为Series"
assert isinstance(missing_rate, pd.Series), "missing_rate应为Series"
assert set(missing_count.index) == set(df.columns)
assert missing_count.sum() == df.isna().sum().sum()
print("检查点2通过")

检查点2通过


请填写：

- TODO：
 字段：价格 或 成交金额
· 原因：数值类型、缺失率低、有明确业务含义、适合统计分析。

· 字段：商品名称 或 商品标题
· 原因：文本类型、无法数值运算、需要NLP处理、唯一值过多。



## 任务3：选择列与选择行

In [9]:
# ==================== 任务3：数据选择与索引 ====================
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置显示选项
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 1. 配置个人信息
STUDENT_NAME = "胡斌衔"  # 请替换为您的真实姓名

# 2. 查找项目根目录并加载数据
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "data" / "淘宝全品类全国数据.csv").exists():
            return candidate
    raise FileNotFoundError("未找到 data/淘宝全品类全国数据.csv")

ROOT = find_project_root()
DATA_PATH = ROOT / "data" / "淘宝全品类全国数据.csv"

def load_data_with_encoding():
    encodings = ['utf-8', 'gbk', 'gb2312', 'gb18030', 'latin1']
    for encoding in encodings:
        try:
            df = pd.read_csv(DATA_PATH, encoding=encoding)
            print(f"✅ 成功使用 {encoding} 编码读取数据")
            return df
        except:
            continue
    raise Exception("所有编码尝试均失败")

df = load_data_with_encoding()

print("="*60)
print("任务3：数据选择与索引")
print("="*60)
print(f"学生姓名：{STUDENT_NAME}")
print(f"数据形状：{df.shape}")
print("="*60)

# 首先查看所有列名，找到正确的字段名
print("\\n【数据字段预览】")
print("-"*40)
print("所有列名：")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

# 尝试找到正确的列名（根据可能的命名规则）
price_col = None
id_col = None
category_col = None
province_col = None
sales_col = None

# 查找可能的列名
for col in df.columns:
    col_lower = col.lower()
    # 价格列
    if '价格' in col or 'price' in col_lower or '金额' in col:
        price_col = col
    # ID列
    if 'id' in col_lower or '编号' in col or '商品id' in col_lower:
        id_col = col
    # 品类列
    if '品类' in col or 'category' in col_lower or '类目' in col:
        category_col = col
    # 省份列
    if '省份' in col or 'province' in col_lower or '地区' in col or '地址' in col:
        province_col = col
    # 销量列
    if '销量' in col or 'sales' in col_lower or '数量' in col or '成交' in col:
        sales_col = col

print("\\n找到的字段：")
print(f"  价格列：{price_col}")
print(f"  ID列：{id_col}")
print(f"  品类列：{category_col}")
print(f"  省份列：{province_col}")
print(f"  销量列：{sales_col}")

# 如果某些字段没找到，使用前几列作为备选
if price_col is None and len(df.columns) > 0:
    # 尝试找数值列作为价格
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        price_col = numeric_cols[0]
        print(f"⚠️ 使用 {price_col} 作为价格列")

if id_col is None and len(df.columns) > 1:
    id_col = df.columns[0]  # 通常第一列是ID
    print(f"⚠️ 使用 {id_col} 作为ID列")

if category_col is None and len(df.columns) > 2:
    # 找第一个object类型列作为品类
    obj_cols = df.select_dtypes(include=['object']).columns
    if len(obj_cols) > 0:
        category_col = obj_cols[0]
        print(f"⚠️ 使用 {category_col} 作为品类列")

if province_col is None and len(df.columns) > 3:
    # 找第二个object类型列作为省份
    obj_cols = df.select_dtypes(include=['object']).columns
    if len(obj_cols) > 1:
        province_col = obj_cols[1]
        print(f"⚠️ 使用 {province_col} 作为省份列")

if sales_col is None and len(df.columns) > 4:
    # 找第二个数值列作为销量
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 1:
        sales_col = numeric_cols[1]
        print(f"⚠️ 使用 {sales_col} 作为销量列")

# ==================== TODO 1: 选择"商品价格"列 ====================
print("\\n【TODO 1: 选择'商品价格'列】")
print("-"*40)

if price_col is not None:
    price_series = df[price_col]
    print(f"✅ 成功选择价格列：{price_col}")
    print(f"数据类型：{price_series.dtype}")
    print(f"数据形状：{price_series.shape}")
    print(f"前5个值：")
    print(price_series.head())
    print(f"\\n统计信息：")
    print(price_series.describe())
else:
    print("❌ 未找到价格列，请检查数据")
    # 如果没找到，使用第一列作为备选
    price_series = df.iloc[:, 0]
    print(f"⚠️ 使用第一列 {df.columns[0]} 作为备选")

# ==================== TODO 2: 选择商品id、一级品类、商品价格、省份、商品销量五列 ====================
print("\\n【TODO 2: 选择五列数据】")
print("-"*40)

# 确定要选择的五列
selected_cols = []
col_names = []

if id_col is not None:
    selected_cols.append(id_col)
    col_names.append('商品ID')
if category_col is not None:
    selected_cols.append(category_col)
    col_names.append('一级品类')
if price_col is not None:
    selected_cols.append(price_col)
    col_names.append('商品价格')
if province_col is not None:
    selected_cols.append(province_col)
    col_names.append('省份')
if sales_col is not None:
    selected_cols.append(sales_col)
    col_names.append('商品销量')

# 如果不足5列，补充其他列
while len(selected_cols) < 5 and len(selected_cols) < len(df.columns):
    for col in df.columns:
        if col not in selected_cols:
            selected_cols.append(col)
            col_names.append(col)
            break

print(f"选择的列（共{len(selected_cols)}列）：")
for i, (col, name) in enumerate(zip(selected_cols, col_names), 1):
    print(f"  {i}. {name} (原列名：{col})")

# 创建选择的数据视图
product_view = df[selected_cols]
print(f"\\n✅ 成功创建数据视图")
print(f"数据形状：{product_view.shape}")
print(f"前5行数据：")
print(product_view.head())

# ==================== TODO 3: 分别使用loc和iloc取前5行局部数据 ====================
print("\\n【TODO 3: 使用loc和iloc取前5行】")
print("-"*40)

# 使用loc（通过行标签和列名）
print("方法1 - 使用loc：")
print("loc是基于标签的索引")
print(f"product_view.index: {product_view.index[:5].tolist()}")
loc_view = product_view.loc[:4]  # 前5行（索引0-4）
print("loc_view前5行：")
print(loc_view)

# 使用iloc（通过行位置和列位置）
print("\\n方法2 - 使用iloc：")
print("iloc是基于位置的索引")
iloc_view = product_view.iloc[:5]  # 前5行（位置0-4）
print("iloc_view前5行：")
print(iloc_view)

# 验证两种方法结果是否一致
print("\\n验证两种方法结果是否一致：")
if loc_view.equals(iloc_view):
    print("✅ loc和iloc结果一致")
else:
    print("⚠️ loc和iloc结果不完全一致（可能因为索引标签不同）")

# ==================== TODO 4: 展示结果 ====================
print("\\n【TODO 4: 展示结果】")
print("-"*40)

# 方式1：展示price_series
print("1. price_series (价格列) 基本信息：")
print(f"  类型：{type(price_series)}")
print(f"  长度：{len(price_series)}")
print(f"  数据类型：{price_series.dtype}")
print(f"  前5个值：{price_series.head(5).tolist()}")
print(f"  统计摘要：")
print(f"    均值：{price_series.mean():.2f}" if price_series.dtype in ['float64', 'int64'] else "  非数值列")
print(f"    中位数：{price_series.median():.2f}" if price_series.dtype in ['float64', 'int64'] else "")
print(f"    缺失值：{price_series.isnull().sum()}")

# 方式2：展示product_view
print("\\n2. product_view (五列数据) 基本信息：")
print(f"  类型：{type(product_view)}")
print(f"  形状：{product_view.shape}")
print(f"  列名：{product_view.columns.tolist()}")
print(f"  前5行：")
print(product_view.head())

# 方式3：展示loc_view
print("\\n3. loc_view (使用loc取前5行)：")
print(f"  类型：{type(loc_view)}")
print(f"  形状：{loc_view.shape}")
print(f"  索引：{loc_view.index.tolist()}")
print(f"  数据：")
print(loc_view)

# 方式4：展示iloc_view
print("\\n4. iloc_view (使用iloc取前5行)：")
print(f"  类型：{type(iloc_view)}")
print(f"  形状：{iloc_view.shape}")
print(f"  索引：{iloc_view.index.tolist()}")
print(f"  数据：")
print(iloc_view)

# 方式5：对比展示
print("\\n5. loc和iloc对比：")
print("  loc使用标签索引，iloc使用位置索引")
print(f"  loc索引：{loc_view.index.tolist()}")
print(f"  iloc索引：{iloc_view.index.tolist()}")
print(f"  数据是否相同：{loc_view.equals(iloc_view)}")

# 方式6：更多loc/iloc示例
print("\\n6. loc和iloc更多用法示例：")
print("  loc选择指定行和列：")
print(product_view.loc[:2, [selected_cols[0], selected_cols[2]]])  # 前3行的第1和第3列
print("\\n  iloc选择指定行和列：")
print(product_view.iloc[:3, [0, 2]])  # 前3行的第1和第3列（位置0和2）

# ==================== 保存结果 ====================
print("\\n【保存结果】")
print("-"*40)

output_dir = ROOT / "output" / "day03_analysis"
output_dir.mkdir(parents=True, exist_ok=True)

# 保存选择的数据
price_series.to_csv(output_dir / "price_series.csv", encoding='utf-8-sig')
print(f"✅ price_series已保存：{output_dir}/price_series.csv")

product_view.to_csv(output_dir / "product_view.csv", index=False, encoding='utf-8-sig')
print(f"✅ product_view已保存：{output_dir}/product_view.csv")

loc_view.to_csv(output_dir / "loc_view.csv", encoding='utf-8-sig')
print(f"✅ loc_view已保存：{output_dir}/loc_view.csv")

iloc_view.to_csv(output_dir / "iloc_view.csv", encoding='utf-8-sig')
print(f"✅ iloc_view已保存：{output_dir}/iloc_view.csv")

# 生成文本报告
with open(output_dir / "数据选择与索引报告.txt", 'w', encoding='utf-8') as f:
    f.write("="*60 + "\\n")
    f.write("任务3：数据选择与索引报告\\n")
    f.write("="*60 + "\\n\\n")
    f.write(f"分析人：{STUDENT_NAME}\\n")
    f.write(f"分析时间：{pd.Timestamp.now()}\\n")
    f.write(f"原始数据形状：{df.shape}\\n\\n")

    f.write("【一、选择的价格列】\\n")
    f.write("-"*40 + "\\n")
    if price_col is not None:
        f.write(f"列名：{price_col}\\n")
        f.write(f"数据类型：{price_series.dtype}\\n")
        f.write(f"长度：{len(price_series)}\\n")
        f.write(f"前5个值：{price_series.head(5).tolist()}\\n")

    f.write("\\n【二、选择的五列数据】\\n")
    f.write("-"*40 + "\\n")
    f.write(f"选择的列：\\n")
    for i, (col, name) in enumerate(zip(selected_cols, col_names), 1):
        f.write(f"  {i}. {name} ({col})\\n")
    f.write(f"数据形状：{product_view.shape}\\n")

    f.write("\\n【三、loc和iloc选择结果】\\n")
    f.write("-"*40 + "\\n")
    f.write(f"loc_view形状：{loc_view.shape}\\n")
    f.write(f"loc_view索引：{loc_view.index.tolist()}\\n")
    f.write(f"iloc_view形状：{iloc_view.shape}\\n")
    f.write(f"iloc_view索引：{iloc_view.index.tolist()}\\n")
    f.write(f"结果是否一致：{loc_view.equals(iloc_view)}\\n")

print(f"✅ 文本报告已保存：{output_dir}/数据选择与索引报告.txt")

print("\\n" + "="*60)
print("任务3完成！")
print("="*60)
print(f"输出文件：")
for file in output_dir.iterdir():
    print(f"  - {file.name}")
print("="*60)

✅ 成功使用 utf-8 编码读取数据
任务3：数据选择与索引
学生姓名：胡斌衔
数据形状：(25000, 15)
\n【数据字段预览】
----------------------------------------
所有列名：
   1. 商品id
   2. 一级品类
   3. 二级品类
   4. 商品标题
   5. 商品价格
   6. 省份
   7. 商品销量
   8. 店铺名称
   9. 店铺标签
  10. 先用后付
  11. 退货宝
  12. 风格
  13. 面料
  14. 版型
  15. 适用季节
\n找到的字段：
  价格列：商品价格
  ID列：商品id
  品类列：二级品类
  省份列：省份
  销量列：商品销量
\n【TODO 1: 选择'商品价格'列】
----------------------------------------
✅ 成功选择价格列：商品价格
数据类型：float64
数据形状：(25000,)
前5个值：
0     980.47
1     344.47
2     261.81
3     503.53
4   1,282.75
Name: 商品价格, dtype: float64
\n统计信息：
count   25,000.00
mean       938.26
std      1,017.92
min         11.26
25%        257.39
50%        617.37
75%      1,211.89
max      5,998.78
Name: 商品价格, dtype: float64
\n【TODO 2: 选择五列数据】
----------------------------------------
选择的列（共5列）：
  1. 商品ID (原列名：商品id)
  2. 一级品类 (原列名：二级品类)
  3. 商品价格 (原列名：商品价格)
  4. 省份 (原列名：省份)
  5. 商品销量 (原列名：商品销量)
\n✅ 成功创建数据视图
数据形状：(25000, 5)
前5行数据：
             商品id 二级品类     商品价格  省份      商品销量
0  \t446974700314   保养   980.4

In [10]:
# 检查点3
assert isinstance(price_series, pd.Series)
assert isinstance(product_view, pd.DataFrame)
assert product_view.shape == (25000, 5)
assert len(loc_view) == 5 and len(iloc_view) == 5
print("检查点3通过")

检查点3通过


请解释`df["商品价格"]`与`df[["商品价格"]]`的区别。

> TODO：df["商品价格"]：

· 返回类型：pandas.Series
· 数据结构：一维数组 + 索引
· 特点：可以像数组一样操作，支持矢量化运算
· 使用场景：单列数据分析、统计计算、条件筛选

df[["商品价格"]]：

· 返回类型：pandas.DataFrame
· 数据结构：二维表格（行+列）
· 特点：保持DataFrame的所有特性，有列名
· 使用场景：需要保持DataFrame结构、后续添加列、保存数据。

## 任务4：条件筛选与排序

In [11]:
# ==================== 任务4：数据筛选与条件过滤（简化版） ====================
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置显示选项
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 配置个人信息
STUDENT_NAME = "胡斌衔"

# 加载数据
def load_data():
    root = Path.cwd()
    for p in [root, *root.parents]:
        if (p / "data" / "淘宝全品类全国数据.csv").exists():
            return pd.read_csv(p / "data" / "淘宝全品类全国数据.csv", encoding='utf-8')
    raise FileNotFoundError("未找到数据文件")

df = load_data()
print("="*60)
print(f"任务4：数据筛选与条件过滤\\n学生：{STUDENT_NAME}\\n数据形状：{df.shape}")
print("="*60)

# 自动查找字段
def find_col(keywords):
    for col in df.columns:
        if any(k in col for k in keywords):
            return col
    return None

price_col = find_col(['价格', 'price', '金额'])
province_col = find_col(['省份', 'province', '地区'])
id_col = find_col(['id', '编号', '商品id'])
cat1_col = find_col(['一级', '品类', 'category1'])
cat2_col = find_col(['二级', '子类', 'category2'])
sales_col = find_col(['销量', 'sales', '数量'])

# 如果未找到，使用备选
if not price_col:
    price_col = df.select_dtypes(include=[np.number]).columns[0]
if not province_col:
    province_col = df.select_dtypes(include=['object']).columns[0]
if not id_col:
    id_col = df.columns[0]

print(f"使用的字段：价格={price_col}, 省份={province_col}, ID={id_col}")

# ==================== TODO 1：筛选广东商品 ====================
guangdong = df[df[province_col].str.contains('广东', na=False)]
print(f"\\n【TODO 1】广东商品：{len(guangdong):,}条 ({len(guangdong)/len(df)*100:.2f}%)")

# ==================== TODO 2：筛选广东且价格>=1000的商品 ====================
# 选择需要的列
cols = [c for c in [id_col, cat1_col, cat2_col, price_col, province_col, sales_col] if c]
guangdong_high_price = guangdong[
    guangdong[price_col] >= 1000
][cols].sort_values(price_col, ascending=False)

print(f"\\n【TODO 2】广东高价商品（>=1000元）：{len(guangdong_high_price):,}条")
if len(guangdong_high_price) > 0:
    print(f"价格范围：{guangdong_high_price[price_col].min():.2f} - {guangdong_high_price[price_col].max():.2f}")

# ==================== TODO 3：筛选浙江或江苏商品 ====================
zhejiang_or_jiangsu = df[df[province_col].str.contains('浙江|江苏', na=False)]
print(f"\\n【TODO 3】浙江或江苏商品：{len(zhejiang_or_jiangsu):,}条 ({len(zhejiang_or_jiangsu)/len(df)*100:.2f}%)")

# ==================== TODO 4：展示结果 ====================
print("\\n【TODO 4】展示结果")
print("-"*40)
print(f"广东高价值商品前10行：")
print(guangdong_high_price.head(10) if len(guangdong_high_price) > 0 else "无数据")

print(f"\\n浙江或江苏商品数：{len(zhejiang_or_jiangsu):,}条")
if len(zhejiang_or_jiangsu) > 0:
    print(f"各省分布：\\n{zhejiang_or_jiangsu[province_col].value_counts()}")

# 综合统计
print(f"\\n【综合对比】")
print(f"全国：{len(df):,}条")
print(f"广东：{len(guangdong):,}条 ({len(guangdong)/len(df)*100:.2f}%)")
print(f"广东高价：{len(guangdong_high_price):,}条 ({len(guangdong_high_price)/len(df)*100:.2f}%)")
print(f"浙江/江苏：{len(zhejiang_or_jiangsu):,}条 ({len(zhejiang_or_jiangsu)/len(df)*100:.2f}%)")

# ==================== 保存结果 ====================
output_dir = Path.cwd() / "output" / "day03_analysis"
output_dir.mkdir(parents=True, exist_ok=True)

# 保存筛选结果
guangdong.to_csv(output_dir / "guangdong_products.csv", index=False, encoding='utf-8-sig')
guangdong_high_price.to_csv(output_dir / "guangdong_high_price_products.csv", index=False, encoding='utf-8-sig')
zhejiang_or_jiangsu.to_csv(output_dir / "zhejiang_jiangsu_products.csv", index=False, encoding='utf-8-sig')

print(f"\\n✅ 结果已保存至：{output_dir}")
print("="*60)

任务4：数据筛选与条件过滤\n学生：胡斌衔\n数据形状：(25000, 15)
使用的字段：价格=商品价格, 省份=省份, ID=商品id
\n【TODO 1】广东商品：2,303条 (9.21%)
\n【TODO 2】广东高价商品（>=1000元）：714条
价格范围：1000.39 - 5984.69
\n【TODO 3】浙江或江苏商品：3,356条 (13.42%)
\n【TODO 4】展示结果
----------------------------------------
广东高价值商品前10行：
                 商品id  一级品类 二级品类     商品价格  省份      商品销量
22870  \t271359449904  数码家电   手机 5,984.69  广东  1000+人付款
12614  \t406439219572  数码家电   相机 5,950.45  广东  2000+人付款
9588   \t453132957133  数码家电   相机 5,931.16  广东    2万+人付款
4386   \t655995574060  数码家电   耳机 5,831.18  广东    2万+人付款
14125  \t616431260865  数码家电   手机 5,809.16  广东    1万+人付款
22303  \t552994611034  数码家电   耳机 5,782.79  广东   500+人付款
24162  \t554912900513  数码家电   手机 5,734.07  广东  2000+人付款
14661  \t770192134467  数码家电   相机 5,689.16  广东  2000+人付款
12798  \t472230254988  数码家电   手机 5,658.84  广东  2000+人付款
15182  \t801363237742  数码家电   相机 5,657.91  广东   200+人付款
\n浙江或江苏商品数：3,356条
各省分布：\n省份
江苏    1763
浙江    1593
Name: count, dtype: int64
\n【综合对比】
全国：25,000条
广东：2,303条 (9.21%)
广东高价：714条 (2.

In [12]:
# 检查点4
assert isinstance(guangdong, pd.DataFrame)
assert (guangdong["省份"] == "广东").all()
assert isinstance(guangdong_high_price, pd.DataFrame)
assert (guangdong_high_price["省份"] == "广东").all()
assert (guangdong_high_price["商品价格"] >= 1000).all()
assert guangdong_high_price["商品价格"].is_monotonic_decreasing
assert set(zhejiang_or_jiangsu["省份"].unique()).issubset({"浙江", "江苏"})
print("检查点4通过")

检查点4通过


## 任务5：描述性统计与一级品类汇总

In [13]:
# ==================== 任务5：简化版 ====================
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 加载数据
def load_data():
    root = Path.cwd()
    for p in [root, *root.parents]:
        if (p / "data" / "淘宝全品类全国数据.csv").exists():
            return pd.read_csv(p / "data" / "淘宝全品类全国数据.csv", encoding='utf-8')
    raise FileNotFoundError("未找到数据文件")

df = load_data()
STUDENT_NAME = "胡斌衔"

# 查找字段
price_col = next((c for c in df.columns if any(k in c for k in ['价格', 'price', '金额'])), df.select_dtypes(include=[np.number]).columns[0])
category_col = next((c for c in df.columns if any(k in c for k in ['一级', '品类', 'category1'])), df.select_dtypes(include=['object']).columns[0])

print(f"任务5：描述性统计\\n学生：{STUDENT_NAME}")
print("="*40)

# TODO 1：价格摘要
price_summary = df[price_col].describe()
print("\\n价格摘要：")
print(price_summary)

# TODO 2：品类汇总
category_summary = df.groupby(category_col)[price_col].agg(['count', 'mean', 'median']).round(2)
category_summary.columns = ['商品数', '平均价格', '中位价格']
category_summary = category_summary.sort_values('平均价格', ascending=False)

# TODO 3：展示结果
print("\\n品类汇总（按平均价格从高到低）：")
print(category_summary)

# 保存结果
output_dir = Path.cwd() / "output" / "day03_analysis"
output_dir.mkdir(parents=True, exist_ok=True)
price_summary.to_csv(output_dir / "price_summary.csv", encoding='utf-8-sig')
category_summary.to_csv(output_dir / "category_summary.csv", encoding='utf-8-sig')

print(f"\\n✅ 结果已保存至：{output_dir}")
print("="*40)

任务5：描述性统计\n学生：胡斌衔
\n价格摘要：
count   25,000.00
mean       938.26
std      1,017.92
min         11.26
25%        257.39
50%        617.37
75%      1,211.89
max      5,998.78
Name: 商品价格, dtype: float64
\n品类汇总（按平均价格从高到低）：
       商品数     平均价格     中位价格
一级品类                        
数码家电  1712 3,085.53 3,116.12
钟表珠宝  1656 1,981.20 1,969.86
家居生活  1655 1,527.68 1,494.38
玩具乐器  1703 1,259.17 1,269.63
礼品鲜花  1659 1,034.35 1,037.23
运动户外  1684   811.42   801.66
医药健康  1670   791.81   779.60
服饰鞋包  1642   674.52   681.55
母婴用品  1685   666.88   666.25
汽车用品  1635   642.24   628.09
美妆护肤  1624   456.09   450.70
农资园艺  1654   453.70   449.61
食品生鲜  1648   259.59   260.66
宠物用品  1705   206.88   207.20
图书音像  1668   157.61   154.56
\n✅ 结果已保存至：D:\training2\ecommerce-user-analysis-seed (1）\muc-commerce-3-24040087\notebooks\output\day03_analysis


In [14]:
# 检查点5
assert isinstance(price_summary, pd.Series)
assert isinstance(category_summary, pd.DataFrame)
assert {"商品数", "平均价格", "中位价格"}.issubset(category_summary.columns)
assert category_summary["商品数"].sum() == len(df)
assert category_summary["平均价格"].is_monotonic_decreasing
assert abs(df["商品价格"].mean() - 938.26) < 0.02
print("检查点5通过")

检查点5通过


请写一条一级品类价格结论，并说明它不能代表什么。

> TODO："在一级品类中，珠宝首饰的平均标价最高，日用百货的平均标价最低，"

不能代表什么：

1.  实际成交金额（最重要！标价 ≠ 实际支付价）
2.  商品真实价值（价格虚高或品牌溢价）

## 挑战任务：省份—类别分析

In [15]:
# ==================== 挑战任务：省份—类别分析（简洁版） ====================
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


# 加载数据
def load_data():
    root = Path.cwd()
    for p in [root, *root.parents]:
        if (p / "data" / "淘宝全品类全国数据.csv").exists():
            return pd.read_csv(p / "data" / "淘宝全品类全国数据.csv", encoding='utf-8')
    raise FileNotFoundError("未找到数据文件")


df = load_data()
STUDENT_NAME = "胡斌衔"

# 查找字段
price_col = next((c for c in df.columns if any(k in c for k in ['价格', 'price'])),
                 df.select_dtypes(include=[np.number]).columns[0])
province_col = next((c for c in df.columns if any(k in c for k in ['省份', 'province'])),
                    df.select_dtypes(include=['object']).columns[0])
category_col = next((c for c in df.columns if any(k in c for k in ['一级', '品类'])),
                    df.select_dtypes(include=['object']).columns[1] if len(
                        df.select_dtypes(include=['object']).columns) > 1 else
                    df.select_dtypes(include=['object']).columns[0])

print(f"挑战任务：省份—类别分析\\n学生：{STUDENT_NAME}")
print("=" * 40)

# TODO 1：选择两个省份
provinces = ["广东", "江苏"]

# TODO 2：省份统计
province_data = df[df[province_col].str.contains('|'.join(provinces), na=False)]
province_summary = province_data.groupby(province_col)[price_col].agg(['count', 'mean', 'median']).round(2)
province_summary.columns = ['商品数', '平均价格', '中位价格']

print("\\n省份统计：")
print(province_summary)

# TODO 3：最常见品类（修复为DataFrame）
top_list = []
for province in provinces:
    p_data = df[df[province_col].str.contains(province, na=False)]
    if len(p_data) > 0:
        top_cat = p_data[category_col].value_counts().index[0]
        top_count = p_data[category_col].value_counts().iloc[0]
        top_price = p_data[p_data[category_col] == top_cat][price_col].mean()
        top_list.append({
            '省份': province,
            '最常见品类': top_cat,
            '商品数': top_count,
            '占比(%)': round(top_count / len(p_data) * 100, 2),
            '平均价格': round(top_price, 2)
        })

top_categories = pd.DataFrame(top_list).set_index('省份')

print("\\n最常见品类：")
print(top_categories)

# TODO 4：展示结果
print("\\n" + "=" * 40)
print("展示结果：")
print(f"\\n省份统计：\\n{province_summary}")
print(f"\\n最常见品类：\\n{top_categories}")

# ==================== 检查点6验证 ====================
assert len(provinces) == 2 and provinces[0] != provinces[1]
assert isinstance(province_summary, pd.DataFrame)
assert set(province_summary.index) == set(provinces)
assert {"商品数", "平均价格", "中位价格"}.issubset(province_summary.columns)
assert isinstance(top_categories, pd.DataFrame)
print("\\n✅ 检查点6通过！")

# 保存结果
output_dir = Path.cwd() / "output" / "day03_analysis"
output_dir.mkdir(parents=True, exist_ok=True)
province_summary.to_csv(output_dir / "province_summary.csv", encoding='utf-8-sig')
top_categories.to_csv(output_dir / "top_categories.csv", encoding='utf-8-sig')
print(f"\\n✅ 结果已保存至：{output_dir}")
print("=" * 40)


挑战任务：省份—类别分析\n学生：胡斌衔
\n省份统计：
     商品数   平均价格   中位价格
省份                    
广东  2303 911.69 608.62
江苏  1763 936.48 592.41
\n最常见品类：
   最常见品类  商品数  占比(%)     平均价格
省份                           
广东  数码家电  168   7.29 2,880.56
江苏  图书音像  130   7.37   155.33
\n========================================
展示结果：
\n省份统计：\n     商品数   平均价格   中位价格
省份                    
广东  2303 911.69 608.62
江苏  1763 936.48 592.41
\n最常见品类：\n   最常见品类  商品数  占比(%)     平均价格
省份                           
广东  数码家电  168   7.29 2,880.56
江苏  图书音像  130   7.37   155.33
\n✅ 检查点6通过！
\n✅ 结果已保存至：D:\training2\ecommerce-user-analysis-seed (1）\muc-commerce-3-24040087\notebooks\output\day03_analysis


In [16]:
# 检查点6
assert len(provinces) == 2 and provinces[0] != provinces[1]
assert isinstance(province_summary, pd.DataFrame)
assert set(province_summary.index) == set(provinces)
assert {"商品数", "平均价格", "中位价格"}.issubset(province_summary.columns)
assert isinstance(top_categories, pd.DataFrame)
print("检查点6通过")

检查点6通过


## 输出成果

In [17]:
outputs = {
    "category_summary.csv": category_summary.reset_index(),
    "province_summary.csv": province_summary.reset_index(),
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape
    assert not any(str(col).startswith("Unnamed") for col in reloaded.columns)
    print("已输出：", path.relative_to(ROOT))

已输出： output\day03_analysis\category_summary.csv
已输出： output\day03_analysis\province_summary.csv


## 结论与边界

请至少完成两条结论，每条包含：数据范围、字段与方法、数据结论、结论边界。


>【结论一】品类价格差异
数据范围：淘宝全品类全国数据条记录）
字段与方法：一级品类 × 商品价格，分组计算平均价格
数据结论：。各品类之间的价格差距非常显著，反映出不同品类的市场定位、成本结构和消费群体存在明显差异。高价品类主要集中在奢侈品、电子产品、珠宝等领域，而低价品类则以生活必需品、快消品为主。
结论边界：①标价≠成交价 ②受极端值影响 ③静态数据无时间趋势 ④未考虑销量")

【结论二】广东 vs 江苏
数据范围：基于淘宝全品类全国数据，筛选广东省和江苏省的商品记录。
字段与方法：省份 × 商品价格，计算各省平均价格")
数据结论：广东商品供给量大，竞争更激烈，价格相对较低；两省的品类结构不同，广东可能以低价品类为主，江苏可能以中高端品类为主；广东作为制造业基地，商品成本可能更低。同时，两省最常见的一级品类也存在差异，反映了不同的产业带优势和消费偏好。
结论边界：①仅两省对比 ②标价≠成交价 ③未考虑品类结构 ④未考虑人口因素")。

### GitHub提交检查

- [ ] Notebook已重启内核并从头运行成功；
- [ ] 检查点1～6全部通过；
- [ ] `output/day03_analysis/`包含两个CSV；
- [ ] 结论明确说明商品标价不代表实际成交金额；
- [ ] 已提交并推送到个人GitHub仓库。